In [10]:
import csv
import gzip
import re
import os
from pathlib import Path

outdir = Path("argweaver_chr2_130_140Mb")

def clean_newick(nhx, sample_names):
    """
    Replace integer leaf labels with sample names and strip NHX annotations.
    Leaves appear after ( or , never after ).
    Internal node labels appear after ).
    """
    n = len(sample_names)

    # Replace leaf labels only — preceded by ( or ,
    def replace_leaf(m):
        idx = int(m.group(1))
        if idx < n:
            return m.group(0).replace(m.group(1), sample_names[idx], 1)
        return m.group(0)  # leave as-is if out of range (shouldn't happen)

    # Leaves are digits preceded by ( or , (with possible whitespace)
    s = re.sub(r'(?<=[(,])(\d+)(?=:)', replace_leaf, nhx)

    # Remove internal node labels: digits immediately after )
    s = re.sub(r'\)(\d+)(?=:|\[|,|\))', ')', s)

    # Remove NHX annotations
    s = re.sub(r'\[&&NHX:[^\]]*\]', '', s)

    # Clean up any doubled colons or empty branch specs
    s = re.sub(r'::+', ':', s)

    return s.strip()


def smc_to_newick_records(smc_gz_path, sample_names):
    opener = gzip.open if str(smc_gz_path).endswith(".gz") else open

    with opener(smc_gz_path, "rt") as f:
        lines = f.readlines()

    region_parts = lines[1].strip().split("\t")
    seq_start = int(region_parts[2])
    seq_end   = int(region_parts[3])

    records = []
    original_index = 0

    for line in lines[2:]:
        if not line.startswith("TREE"):
            continue

        parts  = line.strip().split("\t")
        left   = int(parts[1]) - seq_start
        right  = int(parts[2]) - seq_start + 1
        nhx    = parts[3]

        newick = clean_newick(nhx, sample_names)

        records.append({
            "tree_index": original_index,
            "left":  float(left),
            "right": float(right),
            "mid":   float(0.5 * (left + right)),
            "newick": newick,
        })
        original_index += 1

    return records, seq_start, seq_end


# Read sample names from header
with gzip.open(outdir / "ceu.0.smc.gz", "rt") as f:
    sample_names = f.readline().strip().split("\t")[1:]

print(f"Found {len(sample_names)} samples: {sample_names[:4]} ...")

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

# Sanity check on first tree of first file
with gzip.open(outdir / "ceu.0.smc.gz", "rt") as f:
    for line in f:
        if line.startswith("TREE"):
            nhx = line.strip().split("\t")[3]
            cleaned = clean_newick(nhx, sample_names)
            print("\nFirst cleaned Newick (first 300 chars):")
            print(cleaned[:300])
            break

# Process each posterior sample
for i in range(0, 61, 10):
    infile = outdir / f"ceu.{i}.smc.gz"
    if not infile.exists():
        print(f"skipping {infile}")
        continue

    stem = f"ceu_sample{i}"
    output_tree_path = outdir / f"{stem}.tree"
    output_csv_path  = results_dir / f"{stem}_breaks.csv"

    records, seq_start, seq_end = smc_to_newick_records(infile, sample_names)

    if len(records) == 0:
        print(f"WARNING: no trees found in {infile}")
        continue

    with open(output_tree_path, "w") as f:
        for rec in records:
            f.write(rec["newick"] + "\n")

    with open(output_csv_path, "w", newline="") as cf:
        writer = csv.writer(cf)
        writer.writerow(["file", "tree_index", "left", "right", "mid"])
        for rec in records:
            writer.writerow([
                stem,
                rec["tree_index"],
                rec["left"]  + seq_start,
                rec["right"] + seq_start,
                rec["mid"]   + seq_start,
            ])
            # writer.writerow([stem, rec["tree_index"], rec["left"], rec["right"], rec["mid"]])

    print(f"Wrote {output_tree_path}  ({len(records)} trees)")
    print(f"  first: [{records[0]['left']}, {records[0]['right']})")
    print(f"  last:  [{records[-1]['left']}, {records[-1]['right']})")

Found 40 samples: ['NA06984_1', 'NA07347_2', 'NA07000_2', 'NA06986_2'] ...

First cleaned Newick (first 300 chars):
(((NA06994_1:270.834843,NA11829_1:270.834843):18540.497640,(NA07051_2:8514.177385,((NA06989_2:119.537300,(NA10847_2:119.537300,NA06986_2:119.537300):0.000000):2103.377484,((NA11829_2:1687.298832,(NA11832_2:714.120803,NA07056_2:714.120803):973.178029):535.615952,(NA10851_2:714.120803,(((NA07056_1:270
Wrote argweaver_chr2_130_140Mb/ceu_sample0.tree  (20583 trees)
  first: [0.0, 1440.0)
  last:  [9999839.0, 10000001.0)
Wrote argweaver_chr2_130_140Mb/ceu_sample10.tree  (19453 trees)
  first: [0.0, 35.0)
  last:  [9999839.0, 10000001.0)
Wrote argweaver_chr2_130_140Mb/ceu_sample20.tree  (18802 trees)
  first: [0.0, 120.0)
  last:  [9999839.0, 10000001.0)
Wrote argweaver_chr2_130_140Mb/ceu_sample30.tree  (18762 trees)
  first: [0.0, 1600.0)
  last:  [9998960.0, 10000001.0)
Wrote argweaver_chr2_130_140Mb/ceu_sample40.tree  (18603 trees)
  first: [0.0, 100.0)
  last:  [9999839.0, 